# Healthcare Records Analytics Using Python

## Task 1 — Data Import & Setup

### Step 1 — Import libraries and load the dataset

In [40]:
import pandas as pd
import numpy as np

df = pd.read_csv("healthcare_dataset.csv")
print(df.shape)
df.head()

(55750, 15)


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,sETh cOoKE,27.0,Female,A+,Arthritis,19-01-2023,Stephanie Cunningham,LLC Colon,Aetna,31903.14494,113.0,Elective,04-02-2023,Ibuprofen,Abnormal
1,daVID BerRY,56.0,Male,AB-,Obesity,04-06-2020,Megan Sutton,PLC Gay,medicare,6773.825523,308.0,Emergency,07/04/2020,Paracetamol,Abnormal
2,aMY KIng,25.0,female,B-,Arthritis,14-03-2021,Andrew Velazquez,Floyd-Young,Aetna,16846.10013,153.0,Urgent,12-04-2021,Paracetamol,Abnormal
3,melisSA kNOx,83.0,Female,AB-,Hypertension,07 Jul 2021,Sophia Price,"Mclaughlin and Brown, Haney",UnitedHealthcare,36218.93975,415.0,Emergency,19-07-2021,Ibuprofen,Normal
4,Jack cURRy,71.0,Female,B+,Diabetis,11-04-2020,Richard Martinez,Ellison-Watson,Cigna,49212.96308,106.0,Elective,27-04-2020,Lipitor,Inconclusive


### Step 2 — Inspect current data types

In [41]:
df.info()
print("\nOriginal data types:")
df.dtypes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55750 entries, 0 to 55749
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Name                55748 non-null  object
 1   Age                 54073 non-null  object
 2   Gender              54630 non-null  object
 3   Blood Type          53519 non-null  object
 4   Medical Condition   54626 non-null  object
 5   Date of Admission   55187 non-null  object
 6   Doctor              54052 non-null  object
 7   Hospital            54627 non-null  object
 8   Insurance Provider  54078 non-null  object
 9   Billing Amount      54069 non-null  object
 10  Room Number         54632 non-null  object
 11  Admission Type      55191 non-null  object
 12  Discharge Date      54618 non-null  object
 13  Medication          54634 non-null  object
 14  Test Results        54066 non-null  object
dtypes: object(15)
memory usage: 6.4+ MB

Original data types:


Name                  object
Age                   object
Gender                object
Blood Type            object
Medical Condition     object
Date of Admission     object
Doctor                object
Hospital              object
Insurance Provider    object
Billing Amount        object
Room Number           object
Admission Type        object
Discharge Date        object
Medication            object
Test Results          object
dtype: object

### Step 3 — Convert datatypes

#### Convert Age to numeric

In [42]:
#pandas has loaded almost every column — even the numeric ones — as text.
#Some Age values are plain numbers, some are strings like "33 yrs", "unknown", "thirty"
#Pull out the digits and coerce anything unrecoverable to NaN:

df['Age'] = pd.to_numeric(
    df['Age'].str.extract(r'(\d+\.?\d*)')[0],
    errors='coerce'
)

#### Convert Billing Amount to numeric

In [43]:
#Some entries have a $ sign or say "pending"/"N/A"

df['Billing Amount'] = df['Billing Amount'].str.replace(r'[\$,]', '', regex=True)
df['Billing Amount'] = pd.to_numeric(df['Billing Amount'], errors='coerce')

#### Convert Room Number to numeric

In [44]:
#Some entries look like "Room 328", "450.0", "TBD"

df['Room Number'] = pd.to_numeric(
    df['Room Number'].astype(str).str.extract(r'(\d+)')[0],
    errors='coerce'
)

#### Convert date columns to datetime

In [45]:
#The dates mix several formats (DD-MM-YYYY, YYYY/MM/DD, MM/DD/YYYY, DD Mon YYYY) in the same column. format='mixed' handles all of them in one pass

df['Date of Admission'] = pd.to_datetime(df['Date of Admission'], format='mixed', dayfirst=True, errors='coerce')
df['Discharge Date'] = pd.to_datetime(df['Discharge Date'], format='mixed', dayfirst=True, errors='coerce')

#### Confirm the conversions worked

In [46]:
#Now Age, Billing Amount, Room Number are float64, and both date columns are datetime64.
df.dtypes

Name                          object
Age                          float64
Gender                        object
Blood Type                    object
Medical Condition             object
Date of Admission     datetime64[ns]
Doctor                        object
Hospital                      object
Insurance Provider            object
Billing Amount               float64
Room Number                  float64
Admission Type                object
Discharge Date        datetime64[ns]
Medication                    object
Test Results                  object
dtype: object

### Step 4 — Use a NumPy array for a basic operation

In [47]:
#computing length of stay:

df['Length of Stay'] = (df['Discharge Date'] - df['Date of Admission']).dt.days

length_of_stay = df['Length of Stay'].values.astype(float)

print("Mean(Number of days stayed):",np.nanmean(length_of_stay))
print("Maximum(Number of days stayed):", np.nanmax(length_of_stay))

Mean(Number of days stayed): 15.349732739009008
Maximum(Number of days stayed): 351.0


## Task 2 — Data Cleaning

### Step 1 — Detect missing values

In [48]:
print("Missing Values (before cleaning) ")
print(df.isnull().sum())
print("\nTotal missing cells:", df.isna().sum().sum())

Missing Values (before cleaning) 
Name                     2
Age                   1683
Gender                1120
Blood Type            2231
Medical Condition     1124
Date of Admission      563
Doctor                1698
Hospital              1123
Insurance Provider    1672
Billing Amount        1683
Room Number           1125
Admission Type         559
Discharge Date        1132
Medication            1116
Test Results          1684
Length of Stay        1683
dtype: int64

Total missing cells: 20198


### Step 2 — Standardize categorical labels before deduplicating

In [49]:
#casing inconsistencies of different columns like ("Male" vs "male" vs "MALE")

df['Name'] = df['Name'].str.strip().str.title()

df['Gender'] = df['Gender'].str.strip().str.lower().map(
    {'male': 'Male', 'm': 'Male', 'female': 'Female', 'f': 'Female'})

df['Blood Type'] = df['Blood Type'].str.strip().str.upper()

df['Medical Condition'] = (df['Medical Condition'].str.strip().str.lower()
    .replace({'diabetis': 'Diabetes', 'asthama': 'Asthma'})
    .str.title())

df['Insurance Provider'] = df['Insurance Provider'].str.strip().str.lower().map({
    'blue cross': 'Blue Cross', 'bluecross': 'Blue Cross',
    'united healthcare': 'UnitedHealthcare', 'unitedhealthcare': 'UnitedHealthcare',
    'aetna': 'Aetna', 'cigna': 'Cigna', 'medicare': 'Medicare'})

df['Admission Type'] = df['Admission Type'].str.strip().str.title()
df['Doctor'] = df['Doctor'].str.strip().str.title()
df['Hospital'] = df['Hospital'].str.strip().str.title()
df['Medication'] = df['Medication'].str.strip().str.title()
df['Test Results'] = df['Test Results'].str.strip().str.title()

In [50]:
#Check the cleanup

df['Gender'].value_counts(dropna=False)
df['Insurance Provider'].value_counts(dropna=False)

Insurance Provider
Cigna               10968
Medicare            10880
UnitedHealthcare    10817
Blue Cross          10783
Aetna               10630
NaN                  1672
Name: count, dtype: int64

### Step 3 — Remove exact duplicates

In [51]:
print("Duplicate rows found:", df.duplicated().sum())  
df = df.drop_duplicates().reset_index(drop=True)

Duplicate rows found: 312


### Step 4 — Correct unrealistic values

In [52]:
#Age — negative, zero, or implausibly high values (0, -5, 150, 999) should be treated as invalid, not real ages:
df.loc[(df['Age'] <= 0) | (df['Age'] > 110), 'Age'] = np.nan

In [53]:
#Billing Amount — negative charges and absurd outliers (near $1,000,000) aren't realistic hospital bills:
df.loc[(df['Billing Amount'] <= 0) | (df['Billing Amount'] > 100000), 'Billing Amount'] = np.nan

### Step 5 — Handle missing values (impute vs. drop)

In [61]:
# Numeric — median is robust to skew/outliers
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Billing Amount'] = df['Billing Amount'].fillna(df['Billing Amount'].median())
df['Room Number'] = df['Room Number'].fillna(df['Room Number'].median())

# Categorical — mode
for col in ['Gender', 'Blood Type', 'Medical Condition', 'Insurance Provider',
            'Admission Type', 'Medication', 'Test Results', 'Doctor', 'Hospital']:
    df[col] = df[col].fillna(df[col].mode()[0])

# Critical, non-imputable — drop rows missing Name or both dates
df = df.dropna(subset=['Name'])
df = df.dropna(subset=['Date of Admission'])

#Discharge Date missing (1,120 rows), reconstruct it using Length of Stay

median_stay = df['Length of Stay'].median()   # from your already-computed column
mask = df['Discharge Date'].isna()
df.loc[mask, 'Discharge Date'] = df.loc[mask, 'Date of Admission'] + pd.to_timedelta(median_stay, unit='D')

### Step 6 — Fix impossible length-of-stay values

In [62]:
#2,365 rows have Discharge Date before Date of Admission. Treat those as invalid

df['Length of Stay'] = (df['Discharge Date'] - df['Date of Admission']).dt.days
df.loc[df['Length of Stay'] < 0, 'Length of Stay'] = np.nan
df['Length of Stay'] = df['Length of Stay'].fillna(df['Length of Stay'].median())

### Step 7 — Verify the cleanup

In [63]:
print(df.isnull().sum())
print(df.duplicated().sum())
print(df.describe())

Name                  0
Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Date of Admission     0
Doctor                0
Hospital              0
Insurance Provider    0
Billing Amount        0
Room Number           0
Admission Type        0
Discharge Date        0
Medication            0
Test Results          0
Length of Stay        0
dtype: int64
0
                Age              Date of Admission  Billing Amount  \
count  54844.000000                          54844    54844.000000   
mean      51.543213  2021-11-01 11:59:39.520093440    25607.151879   
min        5.000000            2019-01-06 00:00:00        9.238787   
25%       35.000000            2020-07-28 00:00:00    13746.634875   
50%       52.000000            2021-11-01 00:00:00    25618.411260   
75%       68.000000            2023-02-04 00:00:00    37430.772640   
max       89.000000            2024-12-04 00:00:00    52764.276740   
std       19.309607                      

## Task 3 — Data Exploration using Pandas

### Step 1 — View dataset structure

In [66]:
df.head()
df.tail()
df.shape
df.columns

Index(['Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition',
       'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider',
       'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date',
       'Medication', 'Test Results', 'Length of Stay'],
      dtype='object')

### Step 2 — Check data types and structure

In [67]:
df.info()
df.dtypes

<class 'pandas.core.frame.DataFrame'>
Index: 54844 entries, 0 to 55394
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Name                54844 non-null  object        
 1   Age                 54844 non-null  float64       
 2   Gender              54844 non-null  object        
 3   Blood Type          54844 non-null  object        
 4   Medical Condition   54844 non-null  object        
 5   Date of Admission   54844 non-null  datetime64[ns]
 6   Doctor              54844 non-null  object        
 7   Hospital            54844 non-null  object        
 8   Insurance Provider  54844 non-null  object        
 9   Billing Amount      54844 non-null  float64       
 10  Room Number         54844 non-null  float64       
 11  Admission Type      54844 non-null  object        
 12  Discharge Date      54844 non-null  datetime64[ns]
 13  Medication          54844 non-null  object        


Name                          object
Age                          float64
Gender                        object
Blood Type                    object
Medical Condition             object
Date of Admission     datetime64[ns]
Doctor                        object
Hospital                      object
Insurance Provider            object
Billing Amount               float64
Room Number                  float64
Admission Type                object
Discharge Date        datetime64[ns]
Medication                    object
Test Results                  object
Length of Stay               float64
dtype: object

### Step 3 — Summary statistics

In [69]:
df.describe()                    # numeric columns
df.describe(include='object')    # text/categorical columns

,Name,Gender,Blood Type,Medical Condition,Doctor,Hospital,Insurance Provider,Admission Type,Medication,Test Results
count,54844,54844,54844,54844,54844,54844,54844,54844,54844,54844
unique,39905,2,8,6,39072,38904,5,3,5,3
top,Michael Williams,Female,A+,Diabetes,Michael Smith,Llc Smith,Cigna,Elective,Lipitor,Abnormal
freq,24,28045,8795,10160,1710,1155,12438,18799,11920,19519


### Step 4 — Categorical distributions

In [70]:
#six medical conditions are fairly balanced, there are 3 admission types, 5 insurance providers, 2 genders, 3 test result categories.
df['Medical Condition'].value_counts()
df['Admission Type'].unique()
df.nunique()

Name                  39905
Age                      78
Gender                    2
Blood Type                8
Medical Condition         6
Date of Admission      1878
Doctor                39072
Hospital              38904
Insurance Provider        5
Billing Amount        48122
Room Number             400
Admission Type            3
Discharge Date         1901
Medication                5
Test Results              3
Length of Stay          321
dtype: int64

### Step 5 — groupby summaries

In [73]:
# Avg billing by medical condition
df.groupby('Medical Condition')['Billing Amount'].mean().sort_values(ascending=False)

# Avg length of stay by hospital
df.groupby('Hospital')['Length of Stay'].mean().sort_values(ascending=False)

# Admission counts by insurance provider
df.groupby('Insurance Provider').size().sort_values(ascending=False)

Insurance Provider
Cigna               12438
Medicare            10698
Blue Cross          10629
UnitedHealthcare    10620
Aetna               10459
dtype: int64

## Task 4 — Data Wrangling

### Step 1 — Engineer new fields

In [74]:
# Age Group — bucket ages into standard ranges

bins = [0, 18, 35, 50, 65, 120]
labels = ['0-18', '19-35', '36-50', '51-65', '65+']
df['Age Group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=True)

df['Age Group'].value_counts().sort_index()

Age Group
0-18       849
19-35    13108
36-50    11741
51-65    13573
65+      15573
Name: count, dtype: int64

In [75]:
#Billing Category — tercile-based low/medium/high spend buckets

q1 = df['Billing Amount'].quantile(0.33)
q2 = df['Billing Amount'].quantile(0.66)

def billing_bucket(x):
    if x <= q1:
        return 'Low'
    elif x <= q2:
        return 'Medium'
    return 'High'

df['Billing Category'] = df['Billing Amount'].apply(billing_bucket)
df['Billing Category'].value_counts()

Billing Category
High      18647
Low       18100
Medium    18097
Name: count, dtype: int64

In [76]:
#Cost Per Day — an engagement-score-style composite metric (billing normalized by length of stay)

df['Cost Per Day'] = df['Billing Amount'] / df['Length of Stay'].replace(0, np.nan)
df['Cost Per Day'] = df['Cost Per Day'].fillna(df['Billing Amount'])  # same-day visits: full bill is the daily cost

df['Cost Per Day'].describe()

count    54844.000000
mean      3258.122163
std       5617.744470
min          0.373987
25%        836.246357
50%       1612.027873
75%       3028.432963
max      52211.852970
Name: Cost Per Day, dtype: float64

### Step 2 — Building a hospital-level lookup table and joining it back

In [77]:
hospital_summary = df.groupby('Hospital').agg(
    Hospital_Avg_Billing=('Billing Amount', 'mean'),
    Hospital_Patient_Count=('Name', 'count')
).reset_index()

df = df.merge(hospital_summary, on='Hospital', how='left')

df[['Hospital', 'Billing Amount', 'Hospital_Avg_Billing', 'Hospital_Patient_Count']].head()

,Hospital,Billing Amount,Hospital_Avg_Billing,Hospital_Patient_Count
0,Llc Colon,31903.144940,33395.527840,2
1,Plc Gay,6773.825523,6773.825523,1
2,Floyd-Young,16846.100130,16846.100130,1
3,"Mclaughlin And Brown, Haney",36218.939750,36218.939750,1
4,Ellison-Watson,49212.963080,49212.963080,1


### Step 3 — groupby summaries by Medical Condition, Hospital, and Insurance Provider

In [78]:
# By Medical Condition
df.groupby('Medical Condition').agg(
    Avg_Billing=('Billing Amount', 'mean'),
    Avg_Length_of_Stay=('Length of Stay', 'mean'),
    Patient_Count=('Name', 'count')
).round(2).sort_values('Patient_Count', ascending=False)

# By Hospital (top 10 by volume)
df.groupby('Hospital').agg(
    Avg_Billing=('Billing Amount', 'mean'),
    Patient_Count=('Name', 'count')
).round(2).sort_values('Patient_Count', ascending=False).head(10)

# By Insurance Provider
df.groupby('Insurance Provider').agg(
    Avg_Billing=('Billing Amount', 'mean'),
    Avg_Length_of_Stay=('Length of Stay', 'mean'),
    Patient_Count=('Name', 'count')
).round(2).sort_values('Patient_Count', ascending=False)

,Avg_Billing,Avg_Length_of_Stay,Patient_Count
Insurance Provider,,,
Cigna,25633.08,18.39,12438
Medicare,25705.84,18.42,10698
Blue Cross,25655.88,17.87,10629
UnitedHealthcare,25460.24,18.22,10620
Aetna,25575.04,18.32,10459


## Task 5 — Statistical Analysis

### Step 1 — Mean, median, mode (pandas)

In [80]:
import numpy as np

cols = ['Age', 'Billing Amount', 'Room Number', 'Length of Stay']

for col in cols:
    print(col)
    print("Mean:", df[col].mean())
    print("Median:", df[col].median())
    print("Mode:", df[col].mode()[0])
    print()

Age
Mean: 51.543213478229156
Median: 52.0
Mode: 52.0

Billing Amount
Mean: 25607.15187931097
Median: 25618.41126
Mode: 25618.41126

Room Number
Mean: 300.9189154693312
Median: 302.0
Mode: 302.0

Length of Stay
Mean: 18.248450149514987
Median: 16.0
Mode: 16.0



### Step 2 — Standard deviation and variance

In [81]:
for col in cols:
    print(col, "→ Std:", df[col].std(), "| Variance:", df[col].var())

Age → Std: 19.30960653337665 | Variance: 372.8609044738221
Billing Amount → Std: 13943.016941483991 | Variance: 194407721.4305096
Room Number → Std: 114.01938530494479 | Variance: 13000.42022531746
Length of Stay → Std: 23.198817243493497 | Variance: 538.1851214970112


### Step 3 — Percentiles

In [82]:
for col in cols:
    print(col, df[col].quantile([0.25, 0.5, 0.75, 0.90]))

Age 0.25    35.0
0.50    52.0
0.75    68.0
0.90    79.0
Name: Age, dtype: float64
Billing Amount 0.25    13746.634875
0.50    25618.411260
0.75    37430.772640
0.90    45010.817205
Name: Billing Amount, dtype: float64
Room Number 0.25    204.0
0.50    302.0
0.75    398.0
0.90    459.0
Name: Room Number, dtype: float64
Length of Stay 0.25     9.0
0.50    16.0
0.75    23.0
0.90    28.0
Name: Length of Stay, dtype: float64
